# Simple base-rate merged results

Load multi-model results from downloaded Kaggle Benchmarks runs, or from a local merged CSV.

Each row has **`score`** (`true`/`false`). See `docs/benchmark-design-factors.md` for the simple benchmark: variants (`mc_full`, `data_audit`, `response_audit`), natural condition only, and scoring keys. **`path_c_confusion`** flags answers matching **P(T|C)** (inverse-conditional lure) on `mc_full`.

**Note:** Kaggle runs must use current `example_id`s (`{vignette}__natural__{variant}`). Altered/implausible rows are no longer in `benchmark.csv`. Older runs are skipped at load time.

**Kaggle (all evaluated models):** after `kaggle auth login`:

```powershell
python -m kaggle benchmarks tasks download simple-rate-normative-accuracy `
  -o data/kaggle_runs/simple-rate-normative-accuracy
```

Or: `python scripts/export_simple_rate_kaggle_results.py --download`

Set `LOAD_FROM_KAGGLE = True` in the next cell (default).

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "simple").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Reload so notebook picks up merge/scoring changes without a full kernel restart.
import benchmarks.base_rate as base_rate
import benchmarks.kaggle_runs as kaggle_runs
import benchmarks.simple_rate as simple_rate

importlib.reload(base_rate)
importlib.reload(simple_rate)
importlib.reload(kaggle_runs)

from benchmarks.kaggle_runs import (
    DEFAULT_SIMPLE_RATE_TASK_SLUG,
    download_task_runs,
    merged_simple_results_from_kaggle_runs,
)

print("benchmarks.kaggle_runs:", kaggle_runs.__file__)

LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = True  # True to refresh via Kaggle CLI before loading
KAGGLE_TASK_SLUG = DEFAULT_SIMPLE_RATE_TASK_SLUG
KAGGLE_RUNS_DIR = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
BENCHMARK_CSV = ROOT / "data" / "simple" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "simple"

if LOAD_FROM_KAGGLE:
    if DOWNLOAD_KAGGLE_RUNS:
        download_task_runs(KAGGLE_TASK_SLUG, KAGGLE_RUNS_DIR)
    if not KAGGLE_RUNS_DIR.is_dir():
        raise FileNotFoundError(
            f"Download directory not found: {KAGGLE_RUNS_DIR}\n"
            f"Run: python -m kaggle benchmarks tasks download {KAGGLE_TASK_SLUG} "
            f"-o {KAGGLE_RUNS_DIR}"
        )
    # Filters stale example_ids; raises if no rows match current benchmark.csv.
    merged_rows = merged_simple_results_from_kaggle_runs(
        KAGGLE_RUNS_DIR,
        benchmark_path=BENCHMARK_CSV,
        fill_missing=False,
    )
    df = pd.DataFrame(merged_rows)
    data_source = f"Kaggle runs ({KAGGLE_RUNS_DIR})"
else:
    merged_candidates = sorted(
        MERGED_DIR.glob("simple_merged_results*.csv"),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not merged_candidates:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/simple-benchmark.ipynb."
        )
    MERGED_CSV = merged_candidates[0]
    df = pd.read_csv(MERGED_CSV)
    data_source = str(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
else:
    raise KeyError("Merged data must include 'score'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")
else:
    df["parseable_bool"] = True

if "path_c_confusion" in df.columns:
    df["path_c_confusion_bool"] = (
        df["path_c_confusion"].astype(str).str.lower().eq("true")
    )
else:
    df["path_c_confusion_bool"] = False

df["score_true"] = df["score_value"].astype(bool)

VARIANT_ORDER = ["mc_full", "data_audit", "response_audit"]
CONDITION_ORDER = ["natural"]
INTERSECTION_SIZE_ORDER = ["0", "small", "medium", "large"]
PROBLEM_TYPE_ORDER = ["well_posed"]
SCEPTICISM_REQUIRED_ORDER = ["false", "true"]

EXPECTED_ROWS_PER_MODEL = len(pd.read_csv(BENCHMARK_CSV))
_rows_per_model = df.groupby("model", observed=True).size()
FULL_MODELS = sorted(
    model for model, count in _rows_per_model.items() if count == EXPECTED_ROWS_PER_MODEL
)
PARTIAL_MODELS = sorted(
    model for model, count in _rows_per_model.items() if count != EXPECTED_ROWS_PER_MODEL
)
FULL_MODEL_ORDER = FULL_MODELS
df_full = df.loc[df["model"].isin(FULL_MODELS)].copy()
ROWS_PER_VIGNETTE_PER_MODEL = len(VARIANT_ORDER)

print("Source:", data_source)
print("Rows:", len(df))
n_empty_loaded = int(
    (
        df["llm_response"].isna()
        | df["llm_response"].astype(str).str.strip().eq("")
    ).sum()
)
print("Empty llm_response at load:", n_empty_loaded, "/", len(df))
if LOAD_FROM_KAGGLE and n_empty_loaded:
    raise RuntimeError(
        "Padded empty rows detected — merge should use fill_missing=False. "
        "Check benchmarks/simple_rate.py and re-run this cell."
    )
if not LOAD_FROM_KAGGLE and n_empty_loaded:
    print(
        "Tip: stale padded CSV — set LOAD_FROM_KAGGLE=True or re-export with "
        "scripts/export_simple_rate_kaggle_results.py"
    )
print("Models:", sorted(df["model"].unique()))
print("Full-coverage models:", FULL_MODELS)
if PARTIAL_MODELS:
    print(
        "Excluded partial runs:",
        {model: int(_rows_per_model[model]) for model in PARTIAL_MODELS},
        f"(expected {EXPECTED_ROWS_PER_MODEL} rows each)",
    )
print("Vignettes:", df["vignette_name"].nunique())
print(
    "Normative pass:",
    int(df["score_true"].sum()),
    "/",
    len(df),
    "| P(T|C) confusion:",
    int(df["path_c_confusion_bool"].sum()),
    "/",
    len(df),
)
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df.columns


Index(['example_id', 'vignette_name', 'condition', 'problem_type',
       'intersection_size', 'response_type', 'has_statistics', 'variant',
       'prompt', 'well_posed', 'normative', 'p_c_and_d_given_a', 'p_c', 'p_d',
       'p_t_given_c', 'p_t_given_d', 'normative_choice', 'normative_percent',
       'normative_open', 'confidence_required', 'numeric_score_percent',
       'numeric_score_choice', 'scepticism_required',
       'scepticism_score_target', 'option_a_label', 'option_b_label',
       'option_c_label', 'option_d_label', 'option_e_label', 'option_a_lure',
       'option_b_lure', 'option_c_lure', 'option_d_lure', 'option_e_lure',
       'option_f_label', 'option_g_label', 'option_h_label', 'option_f_lure',
       'option_g_lure', 'option_h_lure', 'model', 'llm_response', 'reasoning',
       'answer_line', 'confidence_line', 'parsed_answer_type',
       'parsed_percent', 'parsed_choice', 'parsed_confidence', 'comment_line',
       'scoring_type', 'parseable', 'score', 'path_c_

## Score by type × model

Rows are the five scored types; columns are full-coverage models. Cells are mean normative **score_pct**.

| Type | Rows used |
|------|-----------|
| Conventional Bayes | natural, disjoint, `mc_full` |
| Flawed Bayes — missing information | natural, overlap, `mc_full` |
| Flawed Bayes — implausible | altered, disjoint, `mc_full` |
| data_audit | all `data_audit` |
| response_audit | all `response_audit` |

In [ ]:
SCORE_TYPE_ORDER = [
    "Conventional Bayes",
    "Flawed Bayes — missing information",
    "Flawed Bayes — implausible",
    "data_audit",
    "response_audit",
]


def assign_score_type(row: pd.Series) -> str | None:
    disjoint = str(row["intersection_size"]) == "0"
    if (
        row["condition"] == "natural"
        and disjoint
        and row["variant"] == "mc_full"
    ):
        return "Conventional Bayes"
    if (
        row["condition"] == "natural"
        and not disjoint
        and row["variant"] == "mc_full"
    ):
        return "Flawed Bayes — missing information"
    if (
        row["condition"] == "altered"
        and disjoint
        and row["variant"] == "mc_full"
    ):
        return "Flawed Bayes — implausible"
    if row["variant"] == "data_audit":
        return "data_audit"
    if row["variant"] == "response_audit":
        return "response_audit"
    return None


def model_score_type_table(data: pd.DataFrame) -> pd.DataFrame:
    work = data.copy()
    work["score_type"] = work.apply(assign_score_type, axis=1)
    work = work.loc[work["score_type"].notna()]
    grouped = work.groupby(["score_type", "model"], observed=True)["score_value"]
    table = (
        grouped.mean()
        .mul(100)
        .round(1)
        .unstack("model")
        .reindex(index=SCORE_TYPE_ORDER, columns=FULL_MODEL_ORDER)
    )
    counts = (
        grouped.size()
        .unstack("model")
        .reindex(index=SCORE_TYPE_ORDER, columns=FULL_MODEL_ORDER)
    )
    print("Rows per type × model:")
    display(counts)
    return table


model_score_type_table(df_full)

Rows per type × model:


model,anthropic/claude-opus-4-1@20250805,anthropic/claude-opus-4-8@default,anthropic/claude-sonnet-4@20250514,google/gemini-3-flash-preview,google/gemini-3.5-flash
score_type,,,,,
Conventional Bayes,11,11,11,11,11
Flawed Bayes — missing information,11,11,11,11,11
Flawed Bayes — implausible,11,11,11,11,11
data_audit,33,33,33,33,33
response_audit,33,33,33,33,33


model,anthropic/claude-opus-4-1@20250805,anthropic/claude-opus-4-8@default,anthropic/claude-sonnet-4@20250514,google/gemini-3-flash-preview,google/gemini-3.5-flash
score_type,,,,,
Conventional Bayes,100.0,100.0,100.0,90.9,90.9
Flawed Bayes — missing information,36.4,100.0,36.4,72.7,90.9
Flawed Bayes — implausible,63.6,63.6,81.8,81.8,90.9
data_audit,93.9,93.9,93.9,93.9,87.9
response_audit,75.8,69.7,60.6,66.7,42.4


## Model ID reference (Anthropic)

Kaggle results use provider-prefixed IDs like `anthropic/claude-haiku-4-5@20251001`. For pre–4.6 Claude models, the part after `@` is a **snapshot date** (`YYYYMMDD`) — a pinned release, not “latest.” `@default` is a platform alias (not a date) that routes to the default endpoint for that model line.

Naming: `claude-{tier}-{major}-{minor}` → e.g. **Haiku 4.5** = fast tier, generation 4, minor version 5. See [Anthropic model IDs](https://platform.claude.com/docs/en/about-claude/models/model-ids-and-versions).

In [ ]:
from IPython.display import Markdown, display

df["empty_llm_response_bool"] = (
    df["llm_response"].isna()
    | df["llm_response"].astype(str).str.strip().eq("")
)
n_empty = int(df["empty_llm_response_bool"].sum())
print(
    f"Empty llm_response: {n_empty} / {len(df)} "
    f"({df['empty_llm_response_bool'].mean() * 100:.1f}%)"
)

if n_empty > 0:

    def empty_response_summary_table(
        group_col: str, *, order: list[str] | None = None
    ) -> pd.DataFrame:
        work = df.copy()
        if group_col == "scepticism_required":
            work[group_col] = work[group_col].astype(str).str.lower()
        grouped = work.groupby(group_col, observed=True)
        summary = pd.DataFrame(
            {
                "n": grouped.size(),
                "empty_llm_response": grouped["empty_llm_response_bool"].sum().astype(int),
            }
        )
        summary["empty_pct"] = (summary["empty_llm_response"] / summary["n"] * 100).round(1)
        if order is not None:
            summary = summary.reindex([value for value in order if value in summary.index])
        elif group_col == "model":
            summary = summary.sort_index()
        return summary

    for title, col, order in [
        ("variant", "variant", VARIANT_ORDER),
        ("problem_type", "problem_type", PROBLEM_TYPE_ORDER),
        ("scepticism_required", "scepticism_required", SCEPTICISM_REQUIRED_ORDER),
        ("intersection_size", "intersection_size", INTERSECTION_SIZE_ORDER),
        ("model", "model", None),
        ("vignette", "vignette_name", sorted(df["vignette_name"].unique())),
    ]:
        display(Markdown(f"### By {title}"))
        display(empty_response_summary_table(col, order=order))

Empty llm_response: 0 / 533 (0.0%)


In [ ]:
def score_summary_table(
    group_col: str,
    *,
    order: list[str] | None = None,
    data: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Counts, parseability mix, normative score, and P(T|C) confusion by group."""
    work = (data if data is not None else df).copy()
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "path_c_confusion": grouped["path_c_confusion_bool"].sum(),
        }
    )
    summary["score_pct"] = (grouped["score_value"].mean() * 100).round(1)
    summary["path_c_pct"] = (grouped["path_c_confusion_bool"].mean() * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])
    return summary


score_summary_table("variant", order=VARIANT_ORDER)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
variant,,,,,,,
mc_prob,59,57,2,0,0,96.6,0.0
mc_w_meta,119,88,28,3,0,73.9,0.0
data_audit,178,166,12,0,0,93.3,0.0
response_audit,177,112,58,7,0,63.3,0.0


## Reformulated problem classes (design table)

| Problem class | Contents | Scoring variant |
|---------------|----------|-----------------|
| **Conventional Bayes** | Real data, disjoint subsets | `mc_full` |
| **Flawed Bayes — missing information** | Real data, intersecting subsets | `mc_full` |

**Mapping:** `natural` + `intersection_size == 0` → Conventional; `natural` + overlap → missing information. Altered/implausible vignettes are not in the current benchmark.

In [ ]:
PROBLEM_CLASS_ORDER = [
    "Conventional Bayes",
    "Flawed Bayes — missing information",
]

PROBLEM_CLASS_SPECS: dict[str, dict[str, str]] = {
    "Conventional Bayes": {
        "contents": "Real data, disjoint subsets",
        "scoring_variant": "mc_full",
    },
    "Flawed Bayes — missing information": {
        "contents": "Real data, intersecting subsets",
        "scoring_variant": "mc_full",
    },
}


def assign_problem_class(row: pd.Series) -> str | None:
    if row["condition"] != "natural":
        return None
    if str(row["intersection_size"]) == "0":
        return "Conventional Bayes"
    return "Flawed Bayes — missing information"


def problem_class_frame(data: pd.DataFrame) -> pd.DataFrame:
    work = data.copy()
    work["problem_class"] = work.apply(assign_problem_class, axis=1)
    return work.loc[work["problem_class"].notna()].copy()


def problem_class_scoring_rows(data: pd.DataFrame) -> pd.DataFrame:
    work = problem_class_frame(data)
    parts: list[pd.DataFrame] = []
    for problem_class, spec in PROBLEM_CLASS_SPECS.items():
        parts.append(
            work.loc[
                (work["problem_class"] == problem_class)
                & (work["variant"] == spec["scoring_variant"])
            ]
        )
    return pd.concat(parts, ignore_index=True)


def model_problem_class_score_table(data: pd.DataFrame) -> pd.DataFrame:
    scored = problem_class_scoring_rows(data)
    rows: list[dict[str, object]] = []
    for model in FULL_MODEL_ORDER:
        row: dict[str, object] = {"model": model}
        for problem_class in PROBLEM_CLASS_ORDER:
            sub = scored.loc[
                (scored["model"] == model) & (scored["problem_class"] == problem_class)
            ]
            row[f"{problem_class}_score_pct"] = (
                round(sub["score_value"].mean() * 100, 1) if len(sub) else None
            )
            row[f"{problem_class}_n"] = int(len(sub))
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")


df_problem_classes = problem_class_frame(df_full)
df_problem_class_scored = problem_class_scoring_rows(df_full)

print(
    "Scored rows per full-coverage model:",
    len(df_problem_class_scored) // len(FULL_MODEL_ORDER),
    "(11 per class × 2 classes)",
)
display(
    pd.DataFrame(
        [
            {
                "problem_class": problem_class,
                "contents": spec["contents"],
                "scoring_variant": spec["scoring_variant"],
                "vignettes": int(
                    df_problem_classes.loc[
                        df_problem_classes["problem_class"] == problem_class,
                        "vignette_name",
                    ].nunique()
                ),
            }
            for problem_class, spec in PROBLEM_CLASS_SPECS.items()
        ]
    ).set_index("problem_class")
)
display(score_summary_table("problem_class", order=PROBLEM_CLASS_ORDER, data=df_problem_class_scored))
model_problem_class_score_table(df_full)

Scored rows per full-coverage model: 22 (11 per class × 2 classes)


,contents,scoring_variant,vignettes
problem_class,,,
Conventional Bayes,"Real data, disjoint subsets",mc_prob,11
Flawed Bayes — missing information,"Real data, intersecting subsets",mc_w_meta,11


,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
problem_class,,,,,,,
Conventional Bayes,55,53,2,0,0,96.4,0.0
Flawed Bayes — missing information,55,37,15,3,0,67.3,0.0


,Conventional Bayes_score_pct,Conventional Bayes_n,Flawed Bayes — missing information_score_pct,Flawed Bayes — missing information_n
model,,,,
anthropic/claude-opus-4-1@20250805,100.0,11,36.4,11
anthropic/claude-opus-4-8@default,100.0,11,100.0,11
anthropic/claude-sonnet-4@20250514,100.0,11,36.4,11
google/gemini-3-flash-preview,90.9,11,72.7,11
google/gemini-3.5-flash,90.9,11,90.9,11


## By problem_type

In [ ]:
score_summary_table(
    "problem_type",
    order=[value for value in PROBLEM_TYPE_ORDER if value in df["problem_type"].unique()],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
problem_type,,,,,,,
well_posed,356,294,54,8,0,82.6,0.0


## By scepticism_required

In [ ]:
score_summary_table(
    "scepticism_required",
    order=[
        value
        for value in SCEPTICISM_REQUIRED_ORDER
        if value in df["scepticism_required"].astype(str).str.lower().unique()
    ],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
scepticism_required,,,,,,,
false,176,153,20,3,0,86.9,0.0
true,357,270,80,7,0,75.6,0.0


## By intersection_size

In [ ]:
score_summary_table(
    "intersection_size",
    order=[value for value in INTERSECTION_SIZE_ORDER if value in df["intersection_size"].unique()],
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
intersection_size,,,,,,,
0,353,282,66,5,0,79.9,0.0
large,180,141,34,5,0,78.3,0.0


In [ ]:
score_summary_table(
    "vignette_name",
    order=sorted(df_full["vignette_name"].unique()),
    data=df_full,
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
vignette_name,,,,,,,
CA Trump voter,30,24,5,1,0,80.0,0.0
HS graduation ACGR (WV vs AZ),30,29,1,0,0,96.7,0.0
NAEP grade 4 reading (MA vs NM),30,27,2,1,0,90.0,0.0
NFL MLB watch attend,15,8,5,2,0,53.3,0.0
book drama streaming,15,11,4,0,0,73.3,0.0
college grad professional job,15,13,2,0,0,86.7,0.0
covid vaccine (blue/red),30,27,3,0,0,90.0,0.0
diabetes insulin obese,15,14,1,0,0,93.3,0.0
discharged weapon (last year),30,20,9,1,0,66.7,0.0


## By model (full coverage only)

Only models with all **88** benchmark rows (`EXPECTED_ROWS_PER_MODEL`). Partial runs are excluded — see load cell output.

In [ ]:
score_summary_table(
    "model",
    order=FULL_MODEL_ORDER,
    data=df_full,
)

,n,score_true,score_false,unparseable,path_c_confusion,score_pct,path_c_pct
model,,,,,,,
anthropic/claude-opus-4-1@20250805,99,78,17,4,0,78.8,0.0
anthropic/claude-opus-4-8@default,99,83,16,0,0,83.8,0.0
anthropic/claude-sonnet-4@20250514,99,75,23,1,0,75.8,0.0
google/gemini-3-flash-preview,99,80,19,0,0,80.8,0.0
google/gemini-3.5-flash,99,73,21,5,0,73.7,0.0


## Score by vignette × model (full coverage)

Mean normative **score_pct** per vignette and model, pooled over all four variants (**4** prompts per cell). **pass** shows `correct/total` for that vignette.

In [ ]:
def vignette_model_score_table() -> pd.DataFrame:
    """Pivot: rows=vignette, columns=(metric, model) with score_pct and pass counts."""
    grouped = df_full.groupby(["vignette_name", "model"], observed=True)["score_value"]
    flat = grouped.agg(
        n="size",
        score_true="sum",
        score_pct=lambda values: round(values.mean() * 100, 1),
    )
    score_pct = flat["score_pct"].unstack("model").reindex(columns=FULL_MODEL_ORDER)
    passes = (
        flat["score_true"].unstack("model").reindex(columns=FULL_MODEL_ORDER).astype(int)
    )
    counts = flat["n"].unstack("model").reindex(columns=FULL_MODEL_ORDER).astype(int)
    pass_frac = passes.astype(str) + "/" + counts.astype(str)
    vignette_order = sorted(df_full["vignette_name"].unique())
    table = pd.concat({"score_pct": score_pct, "pass": pass_frac}, axis=1)
    return table.reindex(vignette_order)


def vignette_model_variant_score_table() -> pd.DataFrame:
    """Pivot: rows=vignette, columns=(model, variant) with score_pct (2 conditions each)."""
    grouped = df_full.groupby(
        ["vignette_name", "model", "variant"], observed=True
    )["score_value"]
    flat = grouped.agg(
        n="size",
        score_pct=lambda values: round(values.mean() * 100, 1),
    )
    score_pct = (
        flat["score_pct"]
        .unstack(["model", "variant"])
        .reindex(columns=pd.MultiIndex.from_product(
            [FULL_MODEL_ORDER, VARIANT_ORDER],
            names=["model", "variant"],
        ))
    )
    vignette_order = sorted(df_full["vignette_name"].unique())
    return score_pct.reindex(vignette_order)


vignette_model_score_table()

score_pct  \
model                             anthropic/claude-opus-4-1@20250805   
vignette_name                                                          
CA Trump voter                                                  83.3   
HS graduation ACGR (WV vs AZ)                                  100.0   
NAEP grade 4 reading (MA vs NM)                                100.0   
NFL MLB watch attend                                            33.3   
book drama streaming                                            33.3   
college grad professional job                                   66.7   
covid vaccine (blue/red)                                       100.0   
diabetes insulin obese                                         100.0   
discharged weapon (last year)                                   66.7   
dog cat household                                               33.3   
english teacher humanities                                     100.0   
fantasy sports (male vs female)                                 83.3   
fantasy sports (under 45 vs male)                               66.7   
healthcare employment                                           83.3   
homeowner suburban mortgage                                     66.7   
homeownership under 35 vs 65                                    83.3   
military overseas (federal pool)                                83.3   
parent married dual income                                      66.7   
physician vs PhD research                                       66.7   
republican gun owner ban                                       100.0   
tax MFJ single CTC                                              66.7   
youtube facebook news                                          100.0   

                                                                     \
model                             anthropic/claude-opus-4-8@default   
vignette_name                                                         
CA Trump voter                                                 66.7   
HS graduation ACGR (WV vs AZ)                                 100.0   
NAEP grade 4 reading (MA vs NM)                               100.0   
NFL MLB watch attend                                          100.0   
book drama streaming                                          100.0   
college grad professional job                                 100.0   
covid vaccine (blue/red)                                       66.7   
diabetes insulin obese                                        100.0   
discharged weapon (last year)                                  66.7   
dog cat household                                             100.0   
english teacher humanities                                    100.0   
fantasy sports (male vs female)                               100.0   
fantasy sports (under 45 vs male)                              66.7   
healthcare employment                                          83.3   
homeowner suburban mortgage                                   100.0   
homeownership under 35 vs 65                                   66.7   
military overseas (federal pool)                               50.0   
parent married dual income                                    100.0   
physician vs PhD research                                      50.0   
republican gun owner ban                                      100.0   
tax MFJ single CTC                                            100.0   
youtube facebook news                                         100.0   

                                                                      \
model                             anthropic/claude-sonnet-4@20250514   
vignette_name                                                          
CA Trump voter                                                  83.3   
HS graduation ACGR (WV vs AZ)                                  100.0   
NAEP grade 4 reading (MA vs NM)                                 66.7   
NFL MLB watch attend                              

In [ ]:
vignette_model_variant_score_table()

model                             anthropic/claude-opus-4-1@20250805  \
variant                                                      mc_prob   
vignette_name                                                          
CA Trump voter                                                 100.0   
HS graduation ACGR (WV vs AZ)                                  100.0   
NAEP grade 4 reading (MA vs NM)                                100.0   
NFL MLB watch attend                                             NaN   
book drama streaming                                             NaN   
college grad professional job                                    NaN   
covid vaccine (blue/red)                                       100.0   
diabetes insulin obese                                           NaN   
discharged weapon (last year)                                  100.0   
dog cat household                                                NaN   
english teacher humanities                                       NaN   
fantasy sports (male vs female)                                100.0   
fantasy sports (under 45 vs male)                                NaN   
healthcare employment                                          100.0   
homeowner suburban mortgage                                      NaN   
homeownership under 35 vs 65                                   100.0   
military overseas (federal pool)                               100.0   
parent married dual income                                       NaN   
physician vs PhD research                                      100.0   
republican gun owner ban                                         NaN   
tax MFJ single CTC                                             100.0   
youtube facebook news                                            NaN   

model                                                                  \
variant                           mc_w_meta data_audit response_audit   
vignette_name                                                           
CA Trump voter                        100.0      100.0           50.0   
HS graduation ACGR (WV vs AZ)         100.0      100.0          100.0   
NAEP grade 4 reading (MA vs NM)       100.0      100.0          100.0   
NFL MLB watch attend                    0.0      100.0            0.0   
book drama streaming                    0.0      100.0            0.0   
college grad professional job           0.0      100.0          100.0   
covid vaccine (blue/red)              100.0      100.0          100.0   
diabetes insulin obese                100.0      100.0          100.0   
discharged weapon (last year)         100.0       50.0           50.0   
dog cat household                       0.0      100.0            0.0   
english teacher humanities            100.0      100.0          100.0   
fantasy sports (male vs female)       100.0      100.0           50.0   
fantasy sports (under 45 vs male)       0.0      100.0          100.0   
healthcare employment                   0.0      100.0          100.0   
homeowner suburban mortgage             0.0      100.0          100.0   
homeownership under 35 vs 65            0.0      100.0          100.0   
military overseas (federal pool)      100.0       50.0          100.0   
parent married dual income              0.0      100.0          100.0   
physician vs PhD research               0.0      100.0           50.0   
republican gun owner ban              100.0      100.0          100.0   
tax MFJ single CTC                      0.0      100.0           50.0   
youtube facebook news                 100.0      100.0          100.0   

model                             anthropic/claude-opus-4-8@default            \
variant                                                     mc_prob mc_w_meta   
vignette_name                                                                   
CA Trump voter                                                100.0       0.0   
HS graduation ACGR (WV vs AZ)                    

## Natural partition: cross-variant agreement (full coverage)

**Subset:** `condition == natural`, `intersection_size == 0` (11 disjoint / well-posed vignettes). Uses `df_full` only — when Opus/Sonnet complete all 88 rows they will appear here automatically.

**mc_full vs audits:** coherence between sceptical **F** and audit scores.

**mc_full vs audits** (scepticism coherence on audit **score**):

| mc_full | audit “agree” when |
|-----------|---------------------|
| **F** | `score == false` |
| **numeric** (A–E) | `score == true` |

In [ ]:
PARTITION_VARIANTS = ("mc_full", "data_audit", "response_audit")

df_partition = df_full.loc[
    (df_full["condition"] == "natural") & (df_full["intersection_size"] == "0")
].copy()
df_partition["score_bool"] = df_partition["score"].astype(str).str.lower().eq("true")


def _mc_choice_upper(choice: object) -> str:
    return str(choice or "").strip().upper()


def mc_full_audit_agree(mc_choice: object, audit_score_bool: bool) -> bool | None:
    """F + audit score false; numeric letter + audit score true."""
    choice = _mc_choice_upper(mc_choice)
    if choice == "F":
        return not bool(audit_score_bool)
    if choice in set("ABCDE"):
        return bool(audit_score_bool)
    return None


def partition_variant_agreement_rows() -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for (model, vignette_name), group in df_partition.groupby(
        ["model", "vignette_name"], observed=True
    ):
        by_variant = {row["variant"]: row for _, row in group.iterrows()}
        if set(by_variant) != set(PARTITION_VARIANTS):
            continue
        mc_full = by_variant["mc_full"]
                data_audit = by_variant["data_audit"]
        response_audit = by_variant["response_audit"]

        full_choice = _mc_choice_upper(mc_full["parsed_choice"])
        
        rows.append(
            {
                "model": model,
                "vignette_name": vignette_name,
                "mc_full_choice": full_choice or None,
                                
                "data_audit_choice": _mc_choice_upper(data_audit["parsed_choice"]) or None,
                "response_audit_choice": _mc_choice_upper(response_audit["parsed_choice"])
                or None,
                "mc_full_data_audit_agree": mc_full_audit_agree(
                    mc_full["parsed_choice"], data_audit["score_bool"]
                ),
                "mc_full_response_audit_agree": mc_full_audit_agree(
                    mc_full["parsed_choice"], response_audit["score_bool"]
                ),
            }
        )
    return pd.DataFrame(rows)


def partition_agreement_summary(data: pd.DataFrame) -> pd.DataFrame:
    metrics = [
        
        "mc_full_data_audit_agree",
        "mc_full_response_audit_agree",
    ]
    rows = []
    for model in FULL_MODEL_ORDER:
        sub = data.loc[data["model"] == model]
        row: dict[str, object] = {"model": model, "vignettes": len(sub)}
        for metric in metrics:
            values = sub[metric].dropna()
            row[metric] = round(values.mean() * 100, 1) if len(values) else None
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")


partition_rows = partition_variant_agreement_rows()
print(
    "Models:", ", ".join(FULL_MODEL_ORDER),
    "| vignettes:", df_partition["vignette_name"].nunique(),
    "| rows:", len(partition_rows),
)
display(partition_agreement_summary(partition_rows))
partition_rows.sort_values(["model", "vignette_name"])

Models: anthropic/claude-opus-4-1@20250805, anthropic/claude-opus-4-8@default, anthropic/claude-sonnet-4@20250514, google/gemini-3-flash-preview, google/gemini-3.5-flash | vignettes: 11 | rows: 0


KeyError: 'model'

## By model × variant (full coverage only)

Mean normative **score_pct** and row count **n** per model and variant. Full coverage per model is **88** rows (22 vignettes × 3 variants).

In [ ]:
def model_variant_score_table() -> pd.DataFrame:
    """Pivot: rows=model, columns=(metric, variant) with score_pct and n."""
    grouped = df_full.groupby(["model", "variant"], observed=True)["score_value"]
    flat = grouped.agg(
        n="size",
        score_pct=lambda values: round(values.mean() * 100, 1),
    )
    score_pct = flat["score_pct"].unstack("variant").reindex(columns=VARIANT_ORDER)
    counts = flat["n"].unstack("variant").reindex(columns=VARIANT_ORDER)
    return (
        pd.concat({"score_pct": score_pct, "n": counts}, axis=1)
        .reindex(FULL_MODEL_ORDER)
        .sort_index()
    )


model_variant_score_table()

score_pct                       \
variant                              mc_prob mc_w_meta data_audit   
model                                                               
anthropic/claude-opus-4-1@20250805      63.6      81.8       90.9   
anthropic/claude-opus-4-8@default       86.4      90.9       95.5   
google/gemini-3-flash-preview           68.2      81.8       95.5   
google/gemini-3.5-flash                 70.5      88.6       90.9   

                                                        n            \
variant                            response_audit mc_prob mc_w_meta   
model                                                                 
anthropic/claude-opus-4-1@20250805           90.9      44        44   
anthropic/claude-opus-4-8@default            72.7      44        44   
google/gemini-3-flash-preview                75.0      44        44   
google/gemini-3.5-flash                      40.9      44        44   

                                                              
variant                            data_audit response_audit  
model                                                         
anthropic/claude-opus-4-1@20250805         44             44  
anthropic/claude-opus-4-8@default          44             44  
google/gemini-3-flash-preview              44             44  
google/gemini-3.5-flash                    44             44

## By model × condition × variant (full coverage only)

Currently only the `natural` condition is in `benchmark.csv`, so this table is a single block per model. Kept for when altered rows return.


In [ ]:
def model_condition_variant_score_table() -> pd.DataFrame:
    """Pivot: rows=(model, condition), columns=(metric, variant) with score_pct and n."""
    grouped = df_full.groupby(["model", "condition", "variant"], observed=True)["score_value"]
    flat = grouped.agg(
        n="size",
        score_pct=lambda values: round(values.mean() * 100, 1),
    )
    score_pct = (
        flat["score_pct"]
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
    )
    counts = (
        flat["n"]
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
    )
    table = pd.concat({"score_pct": score_pct, "n": counts}, axis=1)
    index = pd.MultiIndex.from_product(
        [FULL_MODEL_ORDER, CONDITION_ORDER],
        names=["model", "condition"],
    )
    return table.reindex(index).sort_index()


model_condition_variant_score_table()

score_pct                       \
variant                                        mc_prob mc_w_meta data_audit   
model                              condition                                  
anthropic/claude-opus-4-1@20250805 altered        63.6      95.5       90.9   
                                   natural        63.6      68.2       90.9   
anthropic/claude-opus-4-8@default  altered        90.9      86.4      100.0   
                                   natural        81.8      95.5       90.9   
google/gemini-3-flash-preview      altered        63.6      86.4      100.0   
                                   natural        72.7      77.3       90.9   
google/gemini-3.5-flash            altered        68.2      90.9      100.0   
                                   natural        72.7      86.4       81.8   

                                                                  n            \
variant                                      response_audit mc_prob mc_w_meta   
model                              condition                                    
anthropic/claude-opus-4-1@20250805 altered             90.9      22        22   
                                   natural             90.9      22        22   
anthropic/claude-opus-4-8@default  altered             72.7      22        22   
                                   natural             72.7      22        22   
google/gemini-3-flash-preview      altered             72.7      22        22   
                                   natural             77.3      22        22   
google/gemini-3.5-flash            altered             31.8      22        22   
                                   natural             50.0      22        22   

                                                                        
variant                                      data_audit response_audit  
model                              condition                            
anthropic/claude-opus-4-1@20250805 altered           22             22  
                                   natural           22             22  
anthropic/claude-opus-4-8@default  altered           22             22  
                                   natural           22             22  
google/gemini-3-flash-preview      altered           22             22  
                                   natural           22             22  
google/gemini-3.5-flash            altered           22             22  
                                   natural           22             22

## Altered / implausible (disabled)

The benchmark no longer includes `altered` rows or implausible statistics. Re-enable by rebuilding `data/simple/implausible_*.csv` and setting `include_altered=True` in `build_condition_vignettes()` before regenerating `benchmark.csv`.


In [ ]:
# (no analysis — altered vignettes not in benchmark.csv)


## Unparseable responses

Model and raw response for rows where the answer could not be parsed.

In [ ]:
unparseable = df.loc[~df["parseable_bool"]].sort_values(["model", "example_id"])
print(f"{len(unparseable)} unparseable rows")

pd.set_option("display.max_colwidth", None)
unparseable[["model", "llm_response"]]

In [ ]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]
MC_LURE_COLS = [f"option_{letter}_lure" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, label_col, lure_col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS, MC_LURE_COLS):
        label = row.get(label_col)
        lure = row.get(lure_col)
        if pd.notna(label) and str(label).strip():
            lure_text = f" [{lure}]" if pd.notna(lure) and str(lure).strip() else ""
            parts.append(f"{letter}: {label}{lure_text}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "normative_choice",
        "p_t_given_c",
        "score",
        "score_value",
        "path_c_confusion",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 160)
mc_numeric_probs_view

### Printable comparison

Per vignette: source probabilities from `items.csv` (P(C), P(D), P(T|C), P(T|D)), normative / open / MC answers, and full `mc_numeric_probs` prompt.

In [ ]:
import re

from benchmarks.base_rate import parse_open_response, parse_response

benchmark_df = pd.read_csv(ROOT / "data" / "simple" / "benchmark.csv")


def mc_numeric_options_prompt(prompt: str) -> str:
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    option_lines = [line for line in lines if re.match(r"^[A-E]\.\s", line)]
    return " | ".join(option_lines)


def format_source_ps(item_row: pd.Series) -> str:
    return " | ".join(
        [
            f"P(C)={float(item_row['p_c']):.6g}",
            f"P(D)={float(item_row['p_d']):.6g}",
            f"P(T|C)={float(item_row['p_t_given_c']):.6g}",
            f"P(T|D)={float(item_row['p_t_given_d']):.6g}",
        ]
    )


print_rows: list[dict] = []
for vignette_name in sorted(df["vignette_name"].unique()):
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]
    bench_row = benchmark_df.loc[benchmark_df["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = (
        str(item_mc.get(f"option_{mc_choice.lower()}_label", ""))
        if mc_choice
        else ""
    )
    full_prompt = str(bench_row["prompt"])

    print_rows.append(
        {
            "vignette_name": vignette_name,
            "source_ps": format_source_ps(item_open),
            "normative_pct": float(item_open["normative_percent"]),
            "p_t_given_c_pct": float(item_open["p_t_given_c"]) * 100,
            "open_parsed_pct": parsed_open.percent,
            "mc_label": f"{mc_choice} {mc_label}".strip(),
            "numeric_prompt": mc_numeric_options_prompt(full_prompt),
            "prompt": full_prompt,
        }
    )

print_table = pd.DataFrame(print_rows).sort_values("vignette_name")

print(f"{'vignette_name':<32} {'normative':>10} {'P(T|C)':>10} {'open':>10} {'MC label':>14}")
print("-" * 84)
for row in print_table.itertuples(index=False):
    open_pct = "—" if pd.isna(row.open_parsed_pct) else f"{row.open_parsed_pct:.4g}%"
    print(f"\n{row.vignette_name}")
    print(f"  source Ps:   {row.source_ps}")
    print(f"  normative:   {row.normative_pct:.4g}%  (P(C|T))")
    print(f"  P(T|C):      {row.p_t_given_c_pct:.4g}%  (inverse-conditional lure)")
    print(f"  open parsed: {open_pct}")
    print(f"  MC label:    {row.mc_label}")
    print(f"  numeric prompt: {row.numeric_prompt}")
    print("  prompt:")
    for line in row.prompt.splitlines():
        print(f"    {line}")

print_table.drop(columns=["prompt"])

## Optional: split by model when multiple LLMs are present

In [ ]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["path_c_confusion_bool"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

## `data_audit` by vignette × model

Two tables below (`data_audit` only), sorted by mean score across models (hardest first):

1. **Partition vignettes** — `natural_disjoint` and `altered_disjoint` (two prompts per vignette).
2. **Overlap vignettes** — `natural_overlap` only (one prompt per vignette).

In [ ]:
AUDIT_PROMPT_KIND_ORDER = ("natural_disjoint", "natural_overlap", "altered_disjoint")
AUDIT_PROMPT_KIND_LABEL = {
    "natural_disjoint": "nat_disj",
    "natural_overlap": "nat_ovlp",
    "altered_disjoint": "alt_disj",
}
PARTITION_AUDIT_KINDS = ("natural_disjoint", "altered_disjoint")
OVERLAP_AUDIT_KINDS = ("natural_overlap",)


def _audit_prompt_kind(frame: pd.DataFrame) -> pd.Series:
    disjoint = frame["intersection_size"].astype(str).str.strip().isin(("", "0"))
    natural = frame["condition"].eq("natural")
    altered = frame["condition"].eq("altered")
    kind = pd.Series("other", index=frame.index, dtype="string")
    kind.loc[natural & disjoint] = "natural_disjoint"
    kind.loc[natural & ~disjoint] = "natural_overlap"
    kind.loc[altered & disjoint] = "altered_disjoint"
    return kind


def _mean_score_pct_by_model(score_pct: pd.DataFrame) -> pd.DataFrame:
    kind_cols = score_pct.columns[
        score_pct.columns.get_level_values("model").isin(FULL_MODEL_ORDER)
    ]
    return score_pct[kind_cols].T.groupby(level="model").mean().T


def audit_variant_vignette_model_table(
    data: pd.DataFrame,
    *,
    variant: str,
    audit_kinds: tuple[str, ...],
) -> pd.DataFrame:
    audit = data.loc[data["variant"] == variant].copy()
    audit["audit_kind"] = _audit_prompt_kind(audit)
    audit = audit.loc[audit["audit_kind"].isin(audit_kinds)]
    grouped = audit.groupby(
        ["vignette_name", "model", "audit_kind"], observed=True
    )["score_value"]
    flat = grouped.agg(
        n="size",
        score_true="sum",
        score_pct=lambda values: round(values.mean() * 100, 1),
    )
    kind_columns = pd.MultiIndex.from_product(
        [FULL_MODEL_ORDER, audit_kinds],
        names=["model", "audit_kind"],
    )
    score_pct = flat["score_pct"].unstack(["model", "audit_kind"]).reindex(columns=kind_columns)
    passes = (
        flat["score_true"]
        .unstack(["model", "audit_kind"])
        .reindex(columns=kind_columns)
        .fillna(0)
        .astype(int)
    )
    counts = (
        flat["n"]
        .unstack(["model", "audit_kind"])
        .reindex(columns=kind_columns)
        .fillna(0)
        .astype(int)
    )
    pass_frac = passes.astype(str) + "/" + counts.astype(str)
    pass_frac = pass_frac.where(counts > 0, "")
    pass_frac.columns = pd.MultiIndex.from_tuples(
        [(model, AUDIT_PROMPT_KIND_LABEL[kind]) for model, kind in pass_frac.columns],
        names=["model", "audit_kind"],
    )
    model_means = _mean_score_pct_by_model(score_pct)
    score_pct = score_pct.assign(
        mean_across_models=model_means.mean(axis=1).round(1),
        models_passing=passes.sum(axis=1),
    )
    vignette_order = score_pct["mean_across_models"].sort_values().index.tolist()
    return pd.concat({"score_pct": score_pct, "pass": pass_frac}, axis=1).reindex(vignette_order)


def data_audit_vignette_model_table(
    data: pd.DataFrame,
    *,
    audit_kinds: tuple[str, ...],
) -> pd.DataFrame:
    return audit_variant_vignette_model_table(
        data, variant="data_audit", audit_kinds=audit_kinds
    )


def response_audit_vignette_model_table(
    data: pd.DataFrame,
    *,
    audit_kinds: tuple[str, ...],
) -> pd.DataFrame:
    return audit_variant_vignette_model_table(
        data, variant="response_audit", audit_kinds=audit_kinds
    )


def _print_audit_variant_summary(title: str, table: pd.DataFrame) -> None:
    worst = table["score_pct"]["mean_across_models"]
    model_means = _mean_score_pct_by_model(table["score_pct"])
    all_models_fail = worst.index[(model_means == 0).all(axis=1)]
    print(title)
    print(f"  vignettes: {len(table)}")
    if len(all_models_fail):
        print(f"  failed by all models: {list(all_models_fail)}")
    else:
        print("  no vignette failed for every full-coverage model")
    print("  hardest:")
    display(worst.head(5))


data_audit_partition_table = data_audit_vignette_model_table(
    df_full, audit_kinds=PARTITION_AUDIT_KINDS
)
data_audit_overlap_table = data_audit_vignette_model_table(
    df_full, audit_kinds=OVERLAP_AUDIT_KINDS
)

print(
    "data_audit rows per full-coverage model:",
    int((df_full["variant"] == "data_audit").sum() // len(FULL_MODEL_ORDER)),
)
_print_audit_variant_summary(
    "Partition vignettes (natural_disjoint + altered_disjoint):",
    data_audit_partition_table,
)
data_audit_partition_table

data_audit rows per full-coverage model: 33
Partition vignettes (natural_disjoint + altered_disjoint):
  vignettes: 11
  no vignette failed for every full-coverage model
  hardest:


vignette_name
military overseas (federal pool)     50.0
discharged weapon (last year)        70.0
physician vs PhD research            80.0
CA Trump voter                      100.0
covid vaccine (blue/red)            100.0
Name: mean_across_models, dtype: float64

score_pct  \
model                            anthropic/claude-opus-4-1@20250805   
audit_kind                                         natural_disjoint   
vignette_name                                                         
military overseas (federal pool)                                0.0   
discharged weapon (last year)                                   0.0   
physician vs PhD research                                     100.0   
CA Trump voter                                                100.0   
covid vaccine (blue/red)                                      100.0   
fantasy sports (male vs female)                               100.0   
NAEP grade 4 reading (MA vs NM)                               100.0   
HS graduation ACGR (WV vs AZ)                                 100.0   
homeownership under 35 vs 65                                  100.0   
healthcare employment                                         100.0   
tax MFJ single CTC                                            100.0   

                                                   \
model                                               
audit_kind                       altered_disjoint   
vignette_name                                       
military overseas (federal pool)            100.0   
discharged weapon (last year)               100.0   
physician vs PhD research                   100.0   
CA Trump voter                              100.0   
covid vaccine (blue/red)                    100.0   
fantasy sports (male vs female)             100.0   
NAEP grade 4 reading (MA vs NM)             100.0   
HS graduation ACGR (WV vs AZ)               100.0   
homeownership under 35 vs 65                100.0   
healthcare employment                       100.0   
tax MFJ single CTC                          100.0   

                                                                    \
model                            anthropic/claude-opus-4-8@default   
audit_kind                                        natural_disjoint   
vignette_name                                                        
military overseas (federal pool)                               0.0   
discharged weapon (last year)                                100.0   
physician vs PhD research                                      0.0   
CA Trump voter                                               100.0   
covid vaccine (blue/red)                                     100.0   
fantasy sports (male vs female)                              100.0   
NAEP grade 4 reading (MA vs NM)                              100.0   
HS graduation ACGR (WV vs AZ)                                100.0   
homeownership under 35 vs 65                                 100.0   
healthcare employment                                        100.0   
tax MFJ single CTC                                           100.0   

                                                   \
model                                               
audit_kind                       altered_disjoint   
vignette_name                                       
military overseas (federal pool)            100.0   
discharged weapon (last year)               100.0   
physician vs PhD research                   100.0   
CA Trump voter                              100.0   
covid vaccine (blue/red)                    100.0   
fantasy sports (male vs female)             100.0   
NAEP grade 4 reading (MA vs NM)             100.0   
HS graduation ACGR (WV vs AZ)               100.0   
homeownership under 35 vs 65                100.0   
healthcare employment                       100.0   
tax MFJ single CTC                          100.0   

                                                                     \
model                            anthropic/claude-sonnet-4@20250514   
audit_kind                                         natural_disjoint   
vignette_name                                                         
military overseas (federal pool)                                0

In [ ]:
_print_audit_variant_summary(
    "Overlap vignettes (natural_overlap):",
    data_audit_overlap_table,
)
data_audit_overlap_table

NameError: name '_print_audit_variant_summary' is not defined

## `response_audit` by vignette × model

Same layout as `data_audit` above (`response_audit` only):

1. **Partition vignettes** — `natural_disjoint` and `altered_disjoint`.
2. **Overlap vignettes** — `natural_overlap` only.

In [ ]:
response_audit_partition_table = response_audit_vignette_model_table(
    df_full, audit_kinds=PARTITION_AUDIT_KINDS
)
response_audit_overlap_table = response_audit_vignette_model_table(
    df_full, audit_kinds=OVERLAP_AUDIT_KINDS
)

print(
    "response_audit rows per full-coverage model:",
    int((df_full["variant"] == "response_audit").sum() // len(FULL_MODEL_ORDER)),
)
_print_audit_variant_summary(
    "Partition vignettes (natural_disjoint + altered_disjoint):",
    response_audit_partition_table,
)
response_audit_partition_table

In [ ]:
_print_audit_variant_summary(
    "Overlap vignettes (natural_overlap):",
    response_audit_overlap_table,
)
response_audit_overlap_table